In [1]:
import numpy as np
from scipy.cluster.vq import whiten, kmeans, vq

In [2]:
import glob
import numpy as np
import matplotlib.pyplot as plt

import petlab
import minipet_projector as mpp
from minipet import CSPFile, SLMFile, CLMFile, LOR3DFile
from minipet import N_MODULES, N_PAIRS, N_SIDE, COINC_PAIRS

In [3]:
from minipet import CLMFile
RUN = 'pointken_260929-102942'
clm = CLMFile('pointken_260929-102942.clm.lr5')
coinc = clm.coincidences()                 # prompts + delayed
prompts = clm.coincidences(delayed=False) 
delayed = clm.coincidences(delayed=True)
times = clm.prompt_times()                 # clock ticks of T_CLK_NS
pairs = clm.pair_index(prompts)            # P0..P17 indices
clm.duration_s # Acquisition time for the CLM file in minutes



357.890273907099

In [50]:
coinc.shape

(242923,)

In [4]:
idx_per_s = 242923/357.890273907099
idx_per_05 = idx_per_s * 0.5
idx_per_05
# Divide coinc array into bins of size idx_per_05 to get the number of events in 0.5 seconds
coinc_array_divided = np.array_split(coinc, int(len(coinc)/idx_per_05))

In [ ]:
# Convert coinc_array_divided into RVP sinograms.. not done yet
# TODO: Find LUT for this conversion 

rvp_sinograms = []
for coinc_array in coinc_array_divided:
    rvp_sinogram = mpp.coinc_to_sinogram(clm, lut )
    rvp_sinograms.append(rvp_sinogram)

NameError: name 'lut' is not defined

In [5]:
lors = [LOR3DFile(p) for p in sorted(glob.glob(f'{RUN}.3dlor.lr5'))]

In [7]:
# converting 3D lor to RVP sinogram
geom_bundle = mpp.build_sinogram_projector()
y_rvp = mpp.lors_to_sinogram(*lors[0].crystal_pairs()) # converts the first 3dlor file to an RVP sinogram


In [6]:
def feature_calculation_sino(y_rvp):
    """Feature calculation for sinogram data from RVP data.
    Parameters:
        y_rvp: array of RVP sinogram data -> RVP stands for Radius, View, Plane
    Returns:
        features: array of calculated features
    """ 
    avg_pixel_values = np.mean(y_rvp, axis=(0, 1)) # average pixel values across the axes of the sinogram
    min_pixel_values = np.min(y_rvp, axis=(0, 1))
    max_pixel_values = np.max(y_rvp, axis=(0, 1))
    center_of_mass = np.array([np.sum(np.arange(y_rvp.shape[0]) * np.sum(y_rvp, axis=(1, 2))) / np.sum(y_rvp),
                               np.sum(np.arange(y_rvp.shape[1]) * np.sum(y_rvp, axis=(0, 2))) / np.sum(y_rvp)])
    # Insert center of mass calculation here 
    return avg_pixel_values, min_pixel_values, max_pixel_values, center_of_mass

In [26]:
# PCA for feature reduction
def pca(X, num_components):
    # Center the data
    X_centered = X - np.mean(X, axis=0)
    
    # Compute covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)
    
    # Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # Sort eigenvalues and eigenvectors
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    
    # Select the top 'num_components' eigenvectors
    selected_eigenvectors = sorted_eigenvectors[:, :num_components]
    
    # Project the data onto the new feature space
    X_reduced = np.dot(X_centered, selected_eigenvectors)
    
    return X_reduced, selected_eigenvectors, sorted_eigenvalues[:num_components]

In [38]:
# Make K-Means Clustering Function
def kmeans_clustering(data, num_clusters):
    '''Function for clustering sinogram data with K-means algorithm
    Parameters:
        data: PCA reduced data from the sinograms
        num_clusters: number of clusters to form
        Returns:
        cluster_labels: 1D array of cluster labels for each data point
        centroids: 2D array of cluster centroids'''
    # normalize
    data = whiten(data)
    # perform k-means clustering
    centroids, distortion = kmeans(data, num_clusters)
    # assign each data point to a cluster
    cluster_labels, _ = vq(data, centroids)
    return cluster_labels, centroids, distortion



In [10]:
avg_pixel_values, min_pixel_values, max_pixel_values, center_of_mass = feature_calculation_sino(y_rvp)


In [17]:
center_of_mass

array([ 69.94866251, 104.50553316])

In [21]:
X_arr = np.array([avg_pixel_values, min_pixel_values, max_pixel_values]).T

In [36]:
X_reduced, _, _ = pca(X_arr, num_components=2)
X_reduced = np.vstack([X_reduced, center_of_mass])  # Append center of mass to the reduced features

In [58]:
X_reduced.T

array([[ -0.47615475,   0.52379879,  -0.47615409, ...,  -0.47615504,
         -0.47615495,  69.94866251],
       [  0.23234625,   0.22272663,   0.23234628, ...,   0.23234624,
          0.23234625, 104.50553316]], shape=(2, 1226))

In [59]:
X_reduced_2, _, _ = pca(X_reduced.T, num_components=2)

In [ ]:
# This is the PCA reduced data that can be used for k means
X_reduced_2

array([[-2.10790827e+01,  6.21724894e-15],
       [ 2.10790827e+01, -1.06581410e-14]])

In [40]:
# Perform K-means clustering on the PCA-reduced data for different RVPs (not done yet)